# Phase 2: Virtual Library Expansion, Lipinski Filtering, PAINS Screening & Lead Scoring

**Objective:** Reproduce the canonical Phase 2 pipeline (`phase2_integration.py`) interactively.

**Workflow:**
1. Generate 16 taloside-triazole derivatives (8 building blocks × CuAAC/RuAAC regioisomers)
   — a geometry filter enforces correct triazole topology (1 verified product per BB per template)
2. Apply carbohydrate-adjusted Lipinski filtering (MW ≤600, LogP ≤4, HBD ≤6, HBA ≤12)
3. Screen PAINS alerts via RDKit FilterCatalog (PAINS_A/B/C, 480 entries)
4. Rank Lipinski-passing compounds by composite lead score:

   ```
   lead_score = 0.35*(1-norm_TPSA) + 0.25*(1-norm_MW)
              + 0.15*(1-norm_LogP) + 0.10*(1-norm_RotBonds) + 0.15*SA
   ```

   where SA = 1.0 for 1,4-CuAAC (standard click) and 0.5 for 1,5-RuAAC (Ru catalysis)
5. Export seven CSV files to `phase2_output/`

All library-generation and filtering logic is imported from `taloside_pipeline` — no duplicate implementations in this notebook.

> **Phase 3 note:** Docking against 3ZSJ (AutoDock Vina) is run separately via `phase3_docking.py`.
> After running Phase 2, call `run_phase3_pipeline()` — see `src/taloside_pipeline/phase3_docking.py`.


## 1. Setup & Imports

In [ ]:
import sys
import logging
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from rdkit import Chem
from rdkit.Chem import Crippen, Descriptors, Lipinski

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 10

print("[OK] Standard imports successful")

In [ ]:
# Resolve repository root whether the notebook is launched from project root or notebooks/
repo_root = Path.cwd().resolve()
if not (repo_root / "src").exists() and (repo_root.parent / "src").exists():
    repo_root = repo_root.parent

if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

from taloside_pipeline.descriptor_calculator import load_compounds_from_dict
from taloside_pipeline.glycolibrary_generator import (
    LibraryConfig,
    configure_logging,
    generate_triazole_library,
)
from taloside_pipeline.phase2_integration import (
    apply_lipinski_filter,
    apply_pains_filter,
    compute_lead_scores,
)

logger = configure_logging(log_file=repo_root / "phase2_notebook.log")
logger.setLevel(logging.INFO)

print(f"[OK] Pipeline modules imported (repo root: {repo_root})")

## 2. Scaffold, Building Blocks & Configuration

Constants match `run_phase2_pipeline()` in `phase2_integration.py` exactly.

In [ ]:
SCAFFOLD = (
    "O=C(O[C@H]1[C@@H](OCN=[N+]=[N-])[C@@H](O)[C@@H](CO)O[C@H]1OC)"
    "C4=C([N+]([O-])=O)C=CC=C4"
)

BUILDING_BLOCKS = [
    {"id": "BB-001-Ph", "smiles": "C#Cc1ccccc1"},
    {"id": "BB-002-4OMe", "smiles": "C#Cc1ccc(OC)cc1"},
    {"id": "BB-003-4Cl", "smiles": "C#Cc1ccc(Cl)cc1"},
    {"id": "BB-004-4F", "smiles": "C#Cc1ccc(F)cc1"},
    {"id": "BB-005-3Br", "smiles": "C#Cc1cc(Br)ccc1"},
    {"id": "BB-006-2NO2", "smiles": "C#Cc1ccccc1[N+](=O)[O-]"},
    {"id": "BB-007-Pyridine", "smiles": "C#Cc1ccccn1"},
    {"id": "BB-008-Furan", "smiles": "C#Cc1ccoc1"},
]

config = LibraryConfig(
    max_products=500,
    min_product_mw=250.0,
    max_product_mw=800.0,
    include_stereoisomers=True,
    filter_hypervalent=True,
    output_dir=repo_root / "phase2_output",
)

scaffold_mol = Chem.MolFromSmiles(SCAFFOLD)
assert scaffold_mol is not None, "Invalid scaffold SMILES"

print(f"Scaffold atoms: {scaffold_mol.GetNumAtoms()}")
print(f"Building blocks: {len(BUILDING_BLOCKS)}")
print(f"Output directory: {config.output_dir}")

## 3. Run Canonical Pipeline Steps

In [ ]:
# Step 1: Generate both CuAAC (1,4) and RuAAC (1,5) regioisomers.
# A geometry filter discards products whose triazole ring topology does not match
# the template label, eliminating the wrong-alkyne-orientation artefact and leaving
# exactly 1 correct product per building block per regioisomer (8+8 = 16 total).
library_df = generate_triazole_library(
    scaffold_smiles=SCAFFOLD,
    building_blocks=BUILDING_BLOCKS,
    config=config,
    logger_instance=logger,
)

print(f"Total enumerated products: {len(library_df)}")
print(library_df["regioisomer"].value_counts().to_string())
library_df.head()


In [ ]:
# Step 2: Lipinski filtering (carbohydrate-adjusted thresholds)
lipinski_passed, lipinski_failed = apply_lipinski_filter(library_df, strict_mode=True)

print(f"Lipinski passed: {len(lipinski_passed)} / {len(library_df)}")
print(f"Lipinski failed: {len(lipinski_failed)}")

In [ ]:
# Step 3: PAINS screening on Lipinski-passing compounds
clean_df, pains_df, undetermined_df = apply_pains_filter(lipinski_passed)

print(f"PAINS clean:       {len(clean_df)}")
print(f"PAINS flagged:     {len(pains_df)}")
print(f"PAINS undetermined:{len(undetermined_df)}")

In [ ]:
# Step 4: Lead scoring (same merge logic as phase2_integration.py)
scored_df = compute_lead_scores(lipinski_passed)

pains_status_map = {
    row["compound_id"]: row.get("pains_status", "")
    for _, row in pd.concat([clean_df, pains_df, undetermined_df]).iterrows()
}
if pains_status_map:
    scored_df = scored_df.merge(
        pd.Series(pains_status_map, name="pains_status").rename_axis("compound_id").reset_index(),
        on="compound_id",
        how="left",
    )

print(f"Lead score range: {scored_df['lead_score'].min():.3f} – {scored_df['lead_score'].max():.3f}")
scored_df.head()

In [ ]:
# Step 5: Export all seven CSV files (same names as phase2_integration.py)
config.output_dir.mkdir(parents=True, exist_ok=True)

exports = [
    (library_df, "01_all_generated_compounds.csv"),
    (lipinski_passed, "02_lipinski_passed.csv"),
    (lipinski_failed, "03_lipinski_failed.csv"),
    (clean_df, "04_lipinski_clean_no_pains.csv"),
    (pains_df, "05_lipinski_with_pains.csv"),
    (undetermined_df, "06_pains_undetermined.csv"),
    (scored_df, "07_lead_scored.csv"),
]

for df_export, fname in exports:
    path = config.output_dir / fname
    df_export.to_csv(path, index=False)
    print(f"[OK] {fname} ({len(df_export)} compounds)")

## 4. Descriptive Statistics

In [ ]:
descriptor_cols = [
    "molecular_weight", "logp", "h_donors", "h_acceptors", "tpsa", "rotatable_bonds"
]
print("Library descriptor summary:")
display(library_df[descriptor_cols].describe().round(2))

print("\nProducts per building block:")
display(library_df.groupby(["building_block_id", "regioisomer"]).size().unstack(fill_value=0))

## 5. Visualizations

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
plot_specs = [
    ("molecular_weight", "Molecular Weight (Da)"),
    ("logp", "LogP"),
    ("h_donors", "H-Bond Donors"),
    ("h_acceptors", "H-Bond Acceptors"),
    ("tpsa", "TPSA (Å²)"),
    ("rotatable_bonds", "Rotatable Bonds"),
]

for ax, (col, label) in zip(axes.flat, plot_specs):
    for regio, sub in library_df.groupby("regioisomer"):
        ax.hist(sub[col], bins=8, alpha=0.6, label=regio)
    ax.set_title(label)
    ax.set_xlabel(label)
    ax.legend(fontsize=8)

plt.suptitle("Descriptor Distributions by Regioisomer", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(repo_root / "phase2_output" / "descriptor_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Lipinski pass/fail counts (visualisation only; thresholds match apply_lipinski_filter strict_mode)
thresholds = {"MW": 600, "LogP": 4.0, "HBD": 6, "HBA": 12}
rule_counts = {
    "MW": (library_df["molecular_weight"] <= thresholds["MW"]).sum(),
    "LogP": (library_df["logp"] <= thresholds["LogP"]).sum(),
    "HBD": (library_df["h_donors"] <= thresholds["HBD"]).sum(),
    "HBA": (library_df["h_acceptors"] <= thresholds["HBA"]).sum(),
}

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(rule_counts.keys(), rule_counts.values(), color="steelblue", edgecolor="black")
ax.axhline(len(library_df), color="red", linestyle="--", label="Total library size")
ax.set_ylabel("Compounds passing rule")
ax.set_title("Lipinski Rule Pass Counts (carbohydrate-adjusted thresholds)")
ax.legend()
for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2, int(bar.get_height()), ha="center")
plt.tight_layout()
plt.savefig(repo_root / "phase2_output" / "lipinski_breakdown.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(
    scored_df["molecular_weight"],
    scored_df["logp"],
    c=scored_df["lead_score"],
    cmap="RdYlGn",
    s=120,
    edgecolors="black",
)
plt.colorbar(scatter, ax=ax, label="Lead Score")
for _, row in scored_df.head(5).iterrows():
    ax.annotate(row["compound_id"], (row["molecular_weight"], row["logp"]), fontsize=8)
ax.set_xlabel("Molecular Weight (Da)")
ax.set_ylabel("LogP")
ax.set_title("Lead Score Landscape (Top 5 annotated)")
plt.tight_layout()
plt.savefig(repo_root / "phase2_output" / "lead_score_landscape.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Comparison: Original MSc Talosides vs Generated Library

In [ ]:
original_rows = []
for name, smiles in load_compounds_from_dict().items():
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        continue
    original_rows.append({
        "compound_id": name,
        "type": "Original (MSc)",
        "molecular_weight": Descriptors.MolWt(mol),
        "logp": Crippen.MolLogP(mol),
        "h_donors": Lipinski.NumHDonors(mol),
        "h_acceptors": Lipinski.NumHAcceptors(mol),
        "tpsa": Descriptors.TPSA(mol),
        "rotatable_bonds": Lipinski.NumRotatableBonds(mol),
    })

original_df = pd.DataFrame(original_rows)
generated_summary = library_df[descriptor_cols].mean()
original_summary = original_df[descriptor_cols].mean()

comparison = pd.DataFrame({"Original (n=10)": original_summary, "Generated (n=16)": generated_summary})
comparison["Difference (Gen − Orig)"] = comparison["Generated (n=16)"] - comparison["Original (n=10)"]
display(comparison.round(2))

## 7. Summary

In [ ]:
total = len(library_df)
print("=" * 80)
print("PHASE 2 NOTEBOOK SUMMARY (canonical pipeline workflow)")
print("=" * 80)
print(f"Total enumerated:        {total}")
print(f"  1,4-CuAAC products:    {(library_df['regioisomer'] == '1,4-CuAAC').sum()}")
print(f"  1,5-RuAAC products:    {(library_df['regioisomer'] == '1,5-RuAAC').sum()}")
print(f"Lipinski passed:         {len(lipinski_passed)} ({100 * len(lipinski_passed) / total:.1f}%)")
print(f"Lipinski failed:         {len(lipinski_failed)}")
print(f"PAINS clean:             {len(clean_df)}")
print(f"PAINS flagged:           {len(pains_df)}")
print(f"PAINS undetermined:      {len(undetermined_df)}")
print(f"Lead score range:        {scored_df['lead_score'].min():.3f} – {scored_df['lead_score'].max():.3f}")
print()
print("Top 5 by lead score (SA = synthetic accessibility term):")
for _, row in scored_df.head(5).iterrows():
    sa_label = "CuAAC (SA=1.0)" if row["regioisomer"] == "1,4-CuAAC" else "RuAAC  (SA=0.5)"
    print(f"  {row['compound_id']:42s}  score={row['lead_score']:.3f}  {sa_label}")
print()
print(f"Exports written to: {config.output_dir}")
print()
print("Next step: run phase3_docking.py against 3ZSJ to dock these 14 leads.")


## Appendix: Session Information & Reproducibility

In [ ]:
from rdkit import __version__ as rdkit_version

print("ENVIRONMENT & REPRODUCIBILITY")
print("=" * 80)
print(f"Python version:     {sys.version.split()[0]}")
print(f"RDKit version:      {rdkit_version}")
print(f"Pandas version:     {pd.__version__}")
print(f"NumPy version:      {np.__version__}")
print(f"Matplotlib version: {plt.matplotlib.__version__}")
print(f"Seaborn version:    {sns.__version__}")
print(f"Repository root:    {repo_root}")
print(f"Notebook path:      notebooks/Phase2_VirtualLibraryExpansion.ipynb")
print(f"Pipeline module:    taloside_pipeline.phase2_integration")
print("=" * 80)